# RVC Training — c-elo Kikuyu Voice (fully fixed for Python 3.12 + PyTorch 2.6)
**Run cells top to bottom. Steps 4-6 save to Drive so a disconnect never loses work.**

In [ ]:
# Step 1: Mount Drive + set paths
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE   = '/content/drive/MyDrive/rvc-celo'
DRIVE_CHUNKS = f'{DRIVE_BASE}/dataset'
DRIVE_WORK   = f'{DRIVE_BASE}/work'   # features cached here
DRIVE_OUT    = f'{DRIVE_BASE}/output' # final model saved here
RVC_DIR      = '/content/RVC'
EXP_NAME     = 'celo'
SR           = '40k'
LOG_DIR      = f'{RVC_DIR}/logs/{EXP_NAME}'
os.makedirs(DRIVE_WORK, exist_ok=True)
os.makedirs(DRIVE_OUT,  exist_ok=True)
wavs = [f for f in os.listdir(DRIVE_CHUNKS) if f.endswith('.wav')]
print(f'Drive chunks: {len(wavs)} WAVs')
assert len(wavs) > 10

In [ ]:
# Step 2: Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → T4'
print('GPU:', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# Step 3a: Install deps
import os, sys, importlib, glob, site, urllib.request
if not os.path.exists(RVC_DIR):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print('RVC already cloned')
%cd {RVC_DIR}
!pip install -q librosa soundfile praat-parselmouth ffmpeg-python torchcrepe av pyworld==0.3.4 faiss-gpu tensorboard tqdm scipy scikit-learn bitarray sacrebleu sacremoses sentencepiece
!pip install -q 'omegaconf==2.3.0' 'hydra-core>=1.3'
def _ffs():
    for sp in site.getsitepackages()+['/usr/local/lib/python3.12/dist-packages']:
        p = os.path.join(sp,'fairseq')
        if os.path.isdir(p): return p
    return None
if _ffs() is None:
    !pip install -q --no-deps 'git+https://github.com/facebookresearch/fairseq.git@v0.12.2'
print('fairseq at:', _ffs())

In [ ]:
# Step 3b: Patch fairseq for Python 3.12 + PyTorch 2.6
import os, re, glob, sys, site, importlib, urllib.request
FAIRSEQ_DIR = _ffs()
assert FAIRSEQ_DIR

def _clr(path):
    n = os.path.splitext(os.path.basename(path))[0]
    for f in glob.glob(os.path.join(os.path.dirname(path),'__pycache__',n+'*.pyc')):
        try: os.remove(f)
        except: pass

def _ensure_field(src):
    if re.search(r'from dataclasses import[^\n]*\bfield\b', src): return src
    m = re.search(r'from dataclasses import ([^\n]+)', src)
    if m: return src.replace(m.group(0), m.group(0).rstrip()+', field', 1)
    return 'from dataclasses import field\n'+src

P1 = re.compile(r'^(\s+)(\w+)(\s*:\s*[\w\[\],\. ]+)(\s*=\s*)([A-Z]\w*\([^\n)]*\))(\s*)$', re.MULTILINE)
P2 = re.compile(r'field\(default=([A-Z]\w*\([^\n)]*\))\)')
n1=n2=nf=0
for pyf in glob.glob(f'{FAIRSEQ_DIR}/**/*.py', recursive=True):
    try:
        with open(pyf,encoding='utf-8',errors='ignore') as f: src=f.read()
    except: continue
    orig=src
    src,c2=P2.subn(lambda m: f'field(default_factory=lambda: {m.group(1)})',src)
    src,c1=P1.subn(lambda m: m.group(0) if 'field(' in m.group(0) else f'{m.group(1)}{m.group(2)}{m.group(3)}{m.group(4)}field(default_factory=lambda: {m.group(5)}){m.group(6)}',src)
    if c1+c2==0: continue
    src=_ensure_field(src)
    with open(pyf,'w',encoding='utf-8') as f: f.write(src)
    _clr(pyf); n1+=c1; n2+=c2; nf+=1
print(f'patch A: {n1} bare + {n2} field(default=) fixed in {nf} files')

pb=f'{FAIRSEQ_DIR}/logging/progress_bar.py'
with open(pb) as f: s=f.read()
OLD='try:\n    _tensorboard_writers = {}\n    from torch.utils.tensorboard import SummaryWriter\nexcept ImportError:\n    try:\n        from tensorboardX import SummaryWriter\n    except ImportError:\n        SummaryWriter = None'
NEW='_tensorboard_writers={}\nSummaryWriter=None\ndef _load_sw():\n    global SummaryWriter\n    if SummaryWriter: return\n    try:\n        from torch.utils.tensorboard import SummaryWriter as _S; SummaryWriter=_S\n    except:\n        try:\n            from tensorboardX import SummaryWriter as _S; SummaryWriter=_S\n        except: pass'
s=s.replace(OLD,NEW).replace('    def _writer(self, key):\n        if SummaryWriter is None:\n            return None','    def _writer(self, key):\n        _load_sw()\n        if SummaryWriter is None:\n            return None')
with open(pb,'w') as f: f.write(s); _clr(pb)
print('patch B: progress_bar.py')

ip=f'{FAIRSEQ_DIR}/__init__.py'
with open(ip) as f: s=f.read()
s=s.replace('from fairseq.dataclass.initialize import hydra_init','# hydra_init disabled\ndef hydra_init(*a,**k): pass').replace('\nhydra_init()\n','\n# hydra_init() skipped\n')
with open(ip,'w') as f: f.write(s); _clr(ip)
print('patch C: __init__.py')

hc=f'{FAIRSEQ_DIR}/data/huffman/huffman_coder.py'
if os.path.exists(hc):
    with open(hc) as f: s=f.read()
    s=s.replace('from bitarray import bitarray, util','try:\n    from bitarray import bitarray, util\nexcept ImportError:\n    import subprocess,sys as _s; subprocess.run([_s.executable,"-m","pip","install","-q","bitarray"]); from bitarray import bitarray, util')
    with open(hc,'w') as f: f.write(s); _clr(hc)
    print('patch D: huffman_coder.py')

# Patch E: restore checkpoint_utils.py + usercustomize monkey-patch
cu=f'{FAIRSEQ_DIR}/checkpoint_utils.py'
try:
    with urllib.request.urlopen('https://raw.githubusercontent.com/facebookresearch/fairseq/v0.12.2/fairseq/checkpoint_utils.py',timeout=20) as r:
        orig=r.read().decode('utf-8')
    with open(cu,'w') as f: f.write(orig)
    print('patch E: checkpoint_utils.py restored from GitHub')
except Exception as e:
    print(f'patch E: GitHub restore failed ({e}) — using usercustomize only')
for pc in glob.glob(f'{FAIRSEQ_DIR}/**/__pycache__/checkpoint_utils*.pyc',recursive=True):
    try: os.remove(pc)
    except: pass
uc_dir=site.getusersitepackages(); os.makedirs(uc_dir,exist_ok=True)
with open(os.path.join(uc_dir,'usercustomize.py'),'w') as f:
    f.write('try:\n    import torch as _t;_ol=_t.load\n    def _pl(*a,**k):k.setdefault("weights_only",False);return _ol(*a,**k)\n    _t.load=_pl\nexcept:pass\n')
os.environ['PYTHONPATH']=uc_dir+(':'+os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else '')
print('patch E: usercustomize.py installed')

for k in list(sys.modules): 
    if k.startswith('fairseq'): del sys.modules[k]
importlib.invalidate_caches()
import fairseq,fairseq.checkpoint_utils
print(f'fairseq OK: {fairseq.__version__}')
print('\u2705 Ready')

In [ ]:
# Step 4: Download pretrained weights (skip if already in Drive)
import os
BASE_URL='https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'
for rel in ['pretrained_v2/f0G40k.pth','pretrained_v2/f0D40k.pth','hubert_base.pt','rmvpe.pt']:
    dst=f'{RVC_DIR}/{rel}'; os.makedirs(os.path.dirname(dst),exist_ok=True)
    drv=f'{DRIVE_WORK}/{os.path.basename(rel)}'
    if os.path.exists(drv) and not os.path.exists(dst):
        import shutil; shutil.copy2(drv,dst); print(f'Restored from Drive: {rel}')
    elif not os.path.exists(dst):
        print(f'Downloading {rel}...')
        !wget -q --show-progress -O {dst} {BASE_URL}/{rel}
        import shutil; shutil.copy2(dst,drv)
    else:
        print(f'Already have {rel}')
    sz=os.path.getsize(dst)/1e6
    print(f'  {"\u2713" if sz>0.1 else "\u2717"}  {rel}  ({sz:.1f} MB)')

In [ ]:
# Step 5: Copy dataset chunks into RVC
import os, shutil, glob
from tqdm.notebook import tqdm
DATASET_DEST=f'{RVC_DIR}/dataset/{EXP_NAME}'
os.makedirs(DATASET_DEST,exist_ok=True)
wavs=sorted([f for f in os.listdir(DRIVE_CHUNKS) if f.endswith('.wav')])
for w in tqdm(wavs):
    dst=os.path.join(DATASET_DEST,w)
    if not os.path.exists(dst): shutil.copy2(os.path.join(DRIVE_CHUNKS,w),dst)
print(f'Dataset: {len(os.listdir(DATASET_DEST))} WAVs')

In [ ]:
# Step 6a: Preprocess (resample to 16k)
import os, glob, shutil
%cd {RVC_DIR}
os.makedirs(LOG_DIR,exist_ok=True)

# Restore from Drive if available
drv_16k=f'{DRIVE_WORK}/1_16k_wavs'
loc_16k=f'{LOG_DIR}/1_16k_wavs'
if os.path.exists(drv_16k) and len(glob.glob(f'{drv_16k}/*.wav'))>0 and not os.path.exists(loc_16k):
    shutil.copytree(drv_16k,loc_16k); print(f'Restored 1_16k_wavs from Drive: {len(glob.glob(loc_16k+"/*.wav"))} files')
else:
    !python infer/modules/train/preprocess.py {DATASET_DEST} 40000 2 {LOG_DIR} False 3.0
    if os.path.exists(loc_16k):
        if os.path.exists(drv_16k): shutil.rmtree(drv_16k)
        shutil.copytree(loc_16k,drv_16k)
        print(f'Saved 1_16k_wavs to Drive')

# Also restore/copy 0_gt_wavs
drv_gt=f'{DRIVE_WORK}/0_gt_wavs'
loc_gt=f'{LOG_DIR}/0_gt_wavs'
if os.path.exists(drv_gt) and len(glob.glob(f'{drv_gt}/*.wav'))>0 and not os.path.exists(loc_gt):
    shutil.copytree(drv_gt,loc_gt); print(f'Restored 0_gt_wavs from Drive: {len(glob.glob(loc_gt+"/*.wav"))} files')
elif os.path.exists(loc_gt) and not os.path.exists(drv_gt):
    shutil.copytree(loc_gt,drv_gt); print(f'Saved 0_gt_wavs to Drive')

print(f'16k wavs: {len(glob.glob(loc_16k+"/*.wav"))}')
print(f'GT wavs:  {len(glob.glob(loc_gt+"/*.wav"))}')

In [ ]:
# Step 6b: HuBERT feature extraction
import os, glob, shutil
%cd {RVC_DIR}

drv_feat=f'{DRIVE_WORK}/3_feature768'
loc_feat=f'{LOG_DIR}/3_feature768'
if os.path.exists(drv_feat) and len(glob.glob(f'{drv_feat}/*.npy'))>0 and not os.path.exists(loc_feat):
    shutil.copytree(drv_feat,loc_feat); print(f'Restored features from Drive: {len(glob.glob(loc_feat+"/*.npy"))} files')
else:
    # Ensure hubert is in assets/
    os.makedirs(f'{RVC_DIR}/assets/hubert',exist_ok=True)
    src=f'{RVC_DIR}/hubert_base.pt'; dst=f'{RVC_DIR}/assets/hubert/hubert_base.pt'
    if os.path.exists(src) and not os.path.exists(dst): shutil.copy2(src,dst)
    feat_script=glob.glob(f'{RVC_DIR}/**/extract_feature_print.py',recursive=True)[0]
    !python {feat_script} cuda:0 1 0 0 {LOG_DIR} v2 False
    # Save to Drive
    loc_feat_found=glob.glob(f'{LOG_DIR}/*feature*',recursive=False)
    for d in loc_feat_found:
        if os.path.isdir(d) and glob.glob(f'{d}/*.npy'):
            dn=os.path.basename(d); drv_d=f'{DRIVE_WORK}/{dn}'
            if not os.path.exists(drv_d): shutil.copytree(d,drv_d); print(f'Saved {dn} to Drive')

feats=glob.glob(f'{LOG_DIR}/*feature*/*.npy')
print(f'HuBERT features: {len(feats)}')

In [ ]:
# Step 6c: F0 pitch extraction
import os, glob, shutil
%cd {RVC_DIR}

drv_f0=f'{DRIVE_WORK}/2a_f0'; drv_f0n=f'{DRIVE_WORK}/2b-f0nsf'
loc_f0=f'{LOG_DIR}/2a_f0';   loc_f0n=f'{LOG_DIR}/2b-f0nsf'

if os.path.exists(drv_f0) and len(glob.glob(f'{drv_f0}/*.npy'))>0 and not os.path.exists(loc_f0):
    shutil.copytree(drv_f0,loc_f0); shutil.copytree(drv_f0n,loc_f0n)
    print(f'Restored F0 from Drive: {len(glob.glob(loc_f0+"/*.npy"))} files')
else:
    os.makedirs(f'{RVC_DIR}/assets/rmvpe',exist_ok=True)
    src=f'{RVC_DIR}/rmvpe.pt'; dst=f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'
    if os.path.exists(src) and not os.path.exists(dst): shutil.copy2(src,dst)
    f0_script=glob.glob(f'{RVC_DIR}/**/extract_f0_rmvpe.py',recursive=True)
    if not f0_script: f0_script=glob.glob(f'{RVC_DIR}/**/extract_f0.py',recursive=True)
    !python {f0_script[0]} 1 0 0 {LOG_DIR} True
    for dn in ['2a_f0','2b-f0nsf']:
        src_d=f'{LOG_DIR}/{dn}'; drv_d=f'{DRIVE_WORK}/{dn}'
        if os.path.exists(src_d) and not os.path.exists(drv_d): shutil.copytree(src_d,drv_d); print(f'Saved {dn} to Drive')

print(f'F0 files: {len(glob.glob(loc_f0+"/*.npy"))}')

In [ ]:
# Step 7: Write filelist
import os, glob

def _stem(p): return os.path.basename(p).replace('.npy','').replace('.wav','')

gt_files   = sorted(glob.glob(f'{LOG_DIR}/0_gt_wavs/*.wav'))
feat_files = sorted(glob.glob(f'{LOG_DIR}/*feature*/*.npy'))
f0_files   = sorted(glob.glob(f'{LOG_DIR}/2a_f0/*.npy'))
f0n_files  = sorted(glob.glob(f'{LOG_DIR}/2b-f0nsf/*.npy'))

gt_map   = {_stem(f):f for f in gt_files}
feat_map = {_stem(f):f for f in feat_files}
f0_map   = {_stem(f):f for f in f0_files}
f0n_map  = {_stem(f):f for f in f0n_files}

common = set(gt_map)&set(feat_map)&set(f0_map)&set(f0n_map)
print(f'GT:{len(gt_map)} feat:{len(feat_map)} f0:{len(f0_map)} matched:{len(common)}')
assert len(common)>0

lines=[f'{gt_map[s]}|{feat_map[s]}|{f0_map[s]}|{f0n_map[s]}|0' for s in sorted(common)]
with open(f'{LOG_DIR}/filelist.txt','w') as f: f.write('\n'.join(lines)+'\n')
print(f'Written {len(lines)} entries')
print('Sample:', lines[0])

In [ ]:
# Step 8: Write config.json + Train
import os, json, subprocess, sys
TOTAL_EPOCH=200; BATCH_SIZE=16; SAVE_EVERY=10

cfg={'train':{'log_interval':200,'seed':1234,'epochs':TOTAL_EPOCH,'learning_rate':1e-4,
     'betas':[0.8,0.99],'eps':1e-9,'batch_size':BATCH_SIZE,'fp16_run':True,'lr_decay':0.999875,
     'segment_size':12800,'init_lr_ratio':1,'warmup_epochs':0,'c_mel':45,'c_kl':1.0},
     'data':{'max_wav_value':32768.0,'sampling_rate':40000,'filter_length':2048,'hop_length':400,
     'win_length':2048,'n_mel_channels':125,'mel_fmin':0.0,'mel_fmax':None},
     'model':{'inter_channels':192,'hidden_channels':192,'filter_channels':768,'n_heads':2,
     'n_layers':6,'kernel_size':3,'p_dropout':0,'resblock':'1',
     'resblock_kernel_sizes':[3,7,11],'resblock_dilation_sizes':[[1,3,5],[1,3,5],[1,3,5]],
     'upsample_rates':[10,10,2,2],'upsample_initial_channel':512,
     'upsample_kernel_sizes':[16,16,4,4],'use_spectral_norm':False,'gin_channels':256,'spk_embed_dim':109},
     'version':'v2'}
with open(f'{LOG_DIR}/config.json','w') as f: json.dump(cfg,f,indent=2)
print('config.json written')

# Patch matplotlib tostring_rgb -> buffer_rgba (renamed in matplotlib >= 3.8)
import glob as _gl
for _uf in _gl.glob(f'{RVC_DIR}/**/utils.py', recursive=True):
    with open(_uf, encoding='utf-8') as _f: _us = _f.read()
    if 'tostring_rgb' in _us:
        _us = _us.replace(
            'np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")',
            'np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(*fig.canvas.get_width_height()[::-1], 4)[..., :3].reshape(-1)'
        )
        with open(_uf, 'w', encoding='utf-8') as _f: _f.write(_us)
        print(f'Patched tostring_rgb in {_uf}')

cmd=[sys.executable,f'{RVC_DIR}/infer/modules/train/train.py',
     '-e',EXP_NAME,'-sr',SR,'-f0','1','-bs',str(BATCH_SIZE),'-g','0',
     '-te',str(TOTAL_EPOCH),'-se',str(SAVE_EVERY),
     '-pg',f'{RVC_DIR}/pretrained_v2/f0G40k.pth','-pd',f'{RVC_DIR}/pretrained_v2/f0D40k.pth',
     '-l','0','-c','1','-sw','0','-v','v2']
print('Training started...')
env=os.environ.copy()
proc=subprocess.Popen(cmd,cwd=RVC_DIR,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='')
proc.wait()
print('\nExit code:',proc.returncode)

In [ ]:
# Step 9: Build FAISS index
%cd {RVC_DIR}
!python tools/infer/train-index-v2.py {LOG_DIR} v2
import glob
for f in glob.glob(f'{LOG_DIR}/*.index'): print(f)

In [ ]:
# Step 10: Save model to Drive
import shutil, os, glob
pths=sorted(glob.glob(f'{RVC_DIR}/weights/{EXP_NAME}.pth'))+sorted(glob.glob(f'{LOG_DIR}/G_*.pth'))
idxs=glob.glob(f'{LOG_DIR}/*.index')
if pths:
    shutil.copy2(pths[-1],f'{DRIVE_OUT}/celo.pth'); print('Saved celo.pth')
if idxs:
    shutil.copy2(idxs[0],f'{DRIVE_OUT}/{os.path.basename(idxs[0])}'); print('Saved index')
print('Done — My Drive/rvc-celo/output/')
from google.colab import files
if pths: files.download(f'{DRIVE_OUT}/celo.pth')
if idxs: files.download(f'{DRIVE_OUT}/{os.path.basename(idxs[0])}')